# Steady-state detection for gain validation

The vendor APC gain sheet reports *steady-state* gains: the plant is
sitting still, one handle (MV) is stepped, the plant settles, and the gain is the change in the CV
divided by the change in the MV. Our derived gains have so far been computed over *all* routine data,
which mixes settled operation with upsets, alarms and transients. Before we can claim our method
reproduces the vendor numbers, we have to be able to point at the data and say **"here, and only here,
the plant was actually at steady state"**.

This notebook builds that definition from the data, using the conditions raised in the APC discussion:

| # | Condition | What it removes |
|---|---|---|
| 1a | **frozen / held data** | a sensor stuck on one number - looks perfectly steady, contains no information |
| 1b | **historian ramp-fill** | straight-line values the historian inserts across data gaps (discovered here) |
| 1c | **first-order rate-of-change spikes** | single-sample jumps that immediately reverse |
| 2 | **drift** over the window | the tag is slowly going somewhere |
| 3 | **variability** inside the window | the tag is bouncing around |
| 4 | **SP - PV agreement** | the loop is not actually sitting on its setpoint |

**Scope of the word "steady".** Steadiness is a property of *one tag at one moment*. Every tag gets its
own steady/not-steady timeline - the alarm tag, the handles and the disturbances alike. A *pair*
(MV -> CV) is called steady only when **both** of its tags are steady at the same time. Nothing here
requires the whole plant to be steady at once (Part 6 shows why that never happens).

Everything is read from `DATA/`. No values are imputed or invented anywhere in this notebook.

## 1 - Configuration, vendor pair discovery and data loading

We resolve every non-empty cell of the vendor gain matrix to our historian columns, so the validation
in Part 7 uses **as many MV -> CV pairs as the data supports**, not a hand-picked subset.

In [1]:
import os, re, glob, warnings
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import plotly.graph_objects as go
from plotly.subplots import make_subplots

warnings.filterwarnings('ignore')
pd.set_option('display.width', 260)
pd.set_option('display.max_columns', 80)
pd.set_option('display.max_rows', 300)

DATA = '/home/h604827/ControlActions/DATA'
OUT = '/home/h604827/ControlActions/RESULTS/steady_state_detection'
os.makedirs(OUT, exist_ok=True)

CONFIG = dict(
    NEW_FILE = f'{DATA}/new_rca_pv_op_data/merged_all_historian_tags.parquet',   # 1-min merged history
    OLD_DIR  = f'{DATA}/PV-OP_data',                                             # 8 per-export 1-min files
    SEC30    = f'{DATA}/PV-OP_data/consolidated_30sec_data/all_tags_30sec.parquet',
    EVENTS   = f'{DATA}/combined_events/03LIC_1071_PVLO_PVHI_combined_events.parquet',
    TRIPS    = f'{DATA}/Final_List_Trip_Duration.csv',
    BAND     = f'{DATA}/within_limits_periods.csv',                              # customer 5-95%ile band
    GAINCSV  = f'{DATA}/Gain Data 12Mar26(Sheet1).csv',                          # vendor APC gain matrix
    TARGET   = '03LIC_1071.PV',
    DT_MIN   = 1.0,
)

# ---- detector settings (every threshold is a fraction of the tag's own operating span) ----
SSD = dict(
    W          = 30,     # steadiness window, minutes
    HELD_RUN   = 5,      # identical value repeated this many samples -> stuck
    RAMP_RUN   = 10,     # straight-line run this long -> historian gap-fill
    SPIKE_K    = 8.0,    # rate-of-change spike threshold, robust sigmas
    ALPHA      = 0.02,   # drift across the window  <= 2% of operating span
    BETA       = 0.02,   # std inside the window    <= 2% of operating span
    SP_TOL     = 0.05,   # |PV - SP| <= 5% of operating span
    MIN_LEN    = 30,     # a steady period must last at least this many minutes
)
DT = CONFIG['DT_MIN']
print({k: v for k, v in SSD.items()})

{'W': 30, 'HELD_RUN': 5, 'RAMP_RUN': 10, 'SPIKE_K': 8.0, 'ALPHA': 0.02, 'BETA': 0.02, 'SP_TOL': 0.05, 'MIN_LEN': 30}


In [2]:
# ---- vendor gain matrix -> testable MV/CV pairs ------------------------------------------------
gm = pd.read_csv(CONFIG['GAINCSV'], index_col=0)          # rows = vendor MVs, cols = vendor CVs
print(f'vendor matrix: {gm.shape[0]} MV rows x {gm.shape[1]} CV cols, '
      f'{int(gm.notna().sum().sum())} non-empty gain cells')

# SME-confirmed / fuzzy mappings that the mechanical rule cannot produce
VENDOR_OVERRIDE = {
    '03PIC1141APV': '03PI_1141A.PV',    # SME-confirmed: same pressure, indicator vs controller
    '02FI1000FF':   '02FI_1000.PV',     # a feed-forward signal IS the measured value
    '03FIC3435FF':  '03FIC_3435.PV',
    '03KM0152IPV':  '03KM_0152_I.PV',
    '03PI3154.PV':  '03PI_3154.PV',
    '03TIC1092SP':  '03TIC_1092.PV',    # no .SP in the historian -> loop PV is the SP proxy
    '03TIC1009SP':  '03TIC_1009.PV',
    '03TIC1023SP':  '03TIC_1023.PV',
}
SP_PROXY = {'03TIC1092SP', '03TIC1009SP', '03TIC1023SP'}
FF_AS_PV = {'02FI1000FF', '03FIC3435FF'}

def vendor_to_our(v):
    """Vendor sheet tag name -> our historian column name (None if unmappable)."""
    if v in VENDOR_OVERRIDE:
        return VENDOR_OVERRIDE[v]
    t = str(v).upper().replace(' ', '').replace('_', '').replace('.', '')
    m = re.match(r'^(\d{1,2})([A-Z]+?)(\d+[A-Z]?\d*)(OP|SP|FF|PV|DV)$', t)
    if not m:
        return None
    a, l, n, s = m.groups()
    return f'{a}{l}_{n}.' + ('OP' if s == 'OP' else 'PV')

def instr_key(col):
    """(area, first letter, number) - a controller, its indicator and its valve share this key."""
    t = str(col).upper().replace('_', '').replace('.', '').replace(' ', '')
    m = re.match(r'^(\d{1,2})([A-Z]+?)(\d+[A-Z]?\d*)(OP|PV|SP|DV|FF)?$', t)
    return (m.group(1), m.group(2)[0], m.group(3)) if m else col

# what columns exist anywhere in our sources?
def schema(fp):
    return set(pq.ParquetFile(fp).schema_arrow.names)

AVAIL = schema(CONFIG['NEW_FILE']) | schema(CONFIG['SEC30'])
for fp in sorted(glob.glob(f"{CONFIG['OLD_DIR']}/*.parquet")):
    AVAIL |= schema(fp)

cand = []
for vmv in gm.index:
    for vcv in gm.columns:
        g = gm.at[vmv, vcv]
        if not np.isfinite(g) or g == 0:
            continue
        omv, ocv = vendor_to_our(vmv), vendor_to_our(vcv)
        if omv is None or ocv is None or omv not in AVAIL or ocv not in AVAIL or omv == ocv:
            continue
        proxy = []
        if vmv in SP_PROXY:
            proxy.append('SP->PV proxy')
        if vmv in FF_AS_PV or vcv in FF_AS_PV:
            proxy.append('FF=PV')
        cand.append(dict(vendor_MV=vmv, vendor_CV=vcv, our_MV=omv, our_CV=ocv,
                         vendor_gain=round(float(g), 4), mv_proxy=', '.join(proxy),
                         within_loop=instr_key(omv) == instr_key(ocv)))
PAIRS = pd.DataFrame(cand).drop_duplicates(subset=['our_MV', 'our_CV'])
print(f'\nvendor cells resolvable to our data: {len(PAIRS)} MV->CV pairs '
      f'({int((~PAIRS.within_loop).sum())} independent, {int(PAIRS.within_loop.sum())} within-loop)')
print(f'distinct tags needed: {len(set(PAIRS.our_MV) | set(PAIRS.our_CV))}')

vendor matrix: 53 MV rows x 81 CV cols, 360 non-empty gain cells

vendor cells resolvable to our data: 26 MV->CV pairs (24 independent, 2 within-loop)
distinct tags needed: 22


In [3]:
# ---- 3-tier load: new 1-min merged -> old 1-min per-export -> 30-sec downsampled to 1-min ------
NEED = sorted(set(PAIRS.our_MV) | set(PAIRS.our_CV) | {CONFIG['TARGET'], '03LIC_1071.OP',
                                                       '03LIC_1016.PV', '03LIC_1016.OP',
                                                       '02FI_1000.PV'})
new_cols = schema(CONFIG['NEW_FILE'])
old_files = sorted(glob.glob(f"{CONFIG['OLD_DIR']}/*.parquet"))
old_schema = {fp: schema(fp) for fp in old_files}
sec_cols = schema(CONFIG['SEC30'])

def _read(fp, cols):
    d = pd.read_parquet(fp, columns=['TimeStamp'] + cols)
    d['TimeStamp'] = pd.to_datetime(d['TimeStamp'], errors='coerce')
    return d.dropna(subset=['TimeStamp']).drop_duplicates('TimeStamp').set_index('TimeStamp').sort_index()

frames, prov = [], {}
from_new = [c for c in NEED if c in new_cols]
frames.append(_read(CONFIG['NEW_FILE'], from_new))
prov.update({c: 'new_merged_1min' for c in from_new})

by_src, still = {}, []
for col in [c for c in NEED if c not in new_cols]:
    src = next((fp for fp in old_files if col in old_schema[fp]), None)
    if src:
        by_src.setdefault(src, []).append(col)
        prov[col] = 'old_1min:' + os.path.basename(src)
    else:
        still.append(col)
for src, cols in by_src.items():
    frames.append(_read(src, cols)[cols])

ts = pd.concat(frames, axis=1).asfreq('1min')
from_30 = [c for c in still if c in sec_cols]
if from_30:
    d30 = _read(CONFIG['SEC30'], from_30)
    for c in from_30:
        ts[c] = d30[c].reindex(ts.index)          # 1-min series == the :00-second subset of the 30-sec data
        prov[c] = '30sec->1min'

# ---- trip / shutdown windows are blanked, never treated as steady ----
trips = pd.read_csv(CONFIG['TRIPS']).rename(columns={'Stop Date': 'beg', 'Start Date': 'end'})
twin = trips[['beg', 'end']].apply(pd.to_datetime, errors='coerce').dropna()
twin = twin[twin['end'] > twin['beg']]
idx = ts.index.values
mask_trip = np.zeros(len(ts), bool)
for b, e in twin.itertuples(index=False):
    mask_trip |= (idx >= np.datetime64(b)) & (idx <= np.datetime64(e))
ts.loc[mask_trip, :] = np.nan

TAGS = [c for c in ts.columns if c.endswith(('.PV', '.OP'))]
print(f'grid {ts.shape[0]:,} rows x {ts.shape[1]} cols | {ts.index.min()} -> {ts.index.max()}')
print(f'trip rows blanked {int(mask_trip.sum()):,} | tags resolved {len(TAGS)}/{len(NEED)}')
print('sources:', pd.Series(prov).str.split(':').str[0].value_counts().to_dict())

grid 2,112,525 rows x 26 cols | 2022-01-03 22:45:00 -> 2026-01-09 23:29:00
trip rows blanked 99,890 | tags resolved 26/26
sources: {'new_merged_1min': 24, 'old_1min': 1, '30sec->1min': 1}


## 2 - Condition 1: throw out the data that only *looks* steady

Three kinds of sample carry no process information, and all three are exactly the samples a naive
steadiness test likes most, because they are perfectly smooth.

**1a Held / frozen.** The identical number repeated for 5 minutes or more. On a live analog
measurement that is a stuck sensor or a held historian value. On a valve output a constant value is
normal (a parked valve), so the rule is applied to measurements only.

**1b Historian ramp-fill.** A run where the *second* difference is zero down to float32 storage
precision - i.e. a mathematically exact straight line. Real process data always has curvature at
1-minute resolution. These are the linear interpolations the historian returns across gaps in its
archive. This one was not on the original list; it fell out of the data.

**1c Rate-of-change spikes.** Using the first difference: a jump larger than 8 robust sigmas that
reverses on the very next sample. That is an instrument glitch, not process movement.

In [4]:
EPS32 = float(np.finfo(np.float32).eps)

def op_span(s):
    """Operating span = p95 - p5. Every threshold below is a fraction of this, so one setting works
    for a level in %, a flow in kg/h and a pressure in barg."""
    q = s.quantile([0.05, 0.95])
    return float(q.iloc[1] - q.iloc[0])

def robust_sigma_step(s):
    """Robust sigma of the 1-sample difference (falls back to non-zero diffs when quantised)."""
    a = s.diff().abs().dropna()
    if not len(a):
        return np.nan
    m = a.median()
    if m == 0:
        nz = a[a > 0]
        m = nz.median() if len(nz) else 0.0
    return float(1.4826 * m) if m > 0 else 0.0

def held_mask(s, min_run):
    grp = s.ne(s.shift()).cumsum()
    return ((s.groupby(grp).transform('size') >= min_run) & s.notna()).astype(bool)

def rampfill_mask(s, min_run, c=8.0):
    """Straight-line run at the float32 storage floor that actually moves -> historian gap-fill."""
    lin = ((s.diff().diff().abs() <= c * EPS32 * s.abs().clip(lower=1e-6)) & s.notna()).fillna(False)
    rid = (~lin).cumsum().where(lin)
    g = s.groupby(rid)
    return ((g.transform('size') >= min_run) & (g.transform('max') - g.transform('min') > 0)
            & lin).fillna(False).astype(bool)

def spike_mask(s, k, sigma_step):
    """First-order rate of change: a jump > k sigma that immediately reverses."""
    if not np.isfinite(sigma_step) or sigma_step <= 0:
        return pd.Series(False, index=s.index)
    d = s.diff()
    big = (d.abs() > k * sigma_step).fillna(False).astype(bool)
    rev = (big & big.shift(-1, fill_value=False)
           & (np.sign(d) != np.sign(d.shift(-1)))).fillna(False).astype(bool)
    return (rev | rev.shift(1, fill_value=False)).astype(bool)

QUAL = {}
rows = []
for t in TAGS:
    s = ts[t]
    sig, span = robust_sigma_step(s), op_span(s)
    held = held_mask(s, SSD['HELD_RUN']) if t.endswith('.PV') else pd.Series(False, index=ts.index)
    ramp = rampfill_mask(s, SSD['RAMP_RUN'])
    spk = spike_mask(s, SSD['SPIKE_K'], sig)
    QUAL[t] = dict(span=span, sigma=sig, held=held, ramp=ramp, spike=spk, bad=(held | ramp | spk))
    nv = max(int(s.notna().sum()), 1)
    rows.append(dict(tag=t, valid_pct=round(100 * s.notna().mean(), 1), span=round(span, 3),
                     held_pct=round(100 * held.sum() / nv, 2),
                     rampfill_pct=round(100 * ramp.sum() / nv, 2),
                     spike_pct=round(100 * spk.sum() / nv, 3),
                     discarded_pct=round(100 * QUAL[t]['bad'].sum() / nv, 2)))
qual_tbl = pd.DataFrame(rows).sort_values('discarded_pct', ascending=False)
print('==== data quality per tag (% of the tag\'s own valid samples) ====')
print(qual_tbl.to_string(index=False))

==== data quality per tag (% of the tag's own valid samples) ====
           tag  valid_pct      span  held_pct  rampfill_pct  spike_pct  discarded_pct
 03PIC_1013.OP       94.7    34.849      0.00         50.82      4.725          55.54
 03LIC_1071.PV       94.7    12.956      0.04          0.58      2.919           3.53
 03LIC_1071.OP       94.7    22.246      0.00          0.57      2.713           3.28
 03FIC_3435.PV       91.7 84719.281      2.29          0.68      0.118           3.09
 03PI_1141A.PV       81.3   298.090      2.46          0.54      0.025           3.03
  03TI_1005.PV       94.7    14.052      0.01          0.57      0.915           1.49
 03TIC_1009.PV       94.7    16.851      0.02          0.57      0.740           1.33
 03TIC_1092.PV       94.7     6.602      0.62          0.55      0.013           1.19
  03TI_1108.PV       91.7    17.788      0.35          0.73      0.000           1.08
 03TIC_1009.OP       94.7    25.984      0.00          0.59      0.444    

In [5]:
# ---- FIGURE 1: what historian ramp-fill actually looks like -------------------------------------
T1 = CONFIG['TARGET']
ramp = QUAL[T1]['ramp']
r_idx = ramp[ramp].index
demo_end = r_idx[len(r_idx) // 2]
demo_start = demo_end - pd.Timedelta(minutes=90)
fake = ts.loc[demo_start:demo_end + pd.Timedelta(minutes=90), T1]
real_mid = ts[T1].loc['2024-06-01 12:00']
real = ts.loc['2024-06-01 09:00':'2024-06-01 12:00', T1]

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f'historian ramp-fill  ({demo_start:%Y-%m-%d %H:%M})', 'real process data  (2024-06-01)'))
fig.add_trace(go.Scatter(x=fake.index, y=fake.values, mode='lines+markers',
                         marker=dict(size=3), line=dict(color='orange'), name='gap-fill'), 1, 1)
fig.add_trace(go.Scatter(x=real.index, y=real.values, mode='lines+markers',
                         marker=dict(size=3), line=dict(color='steelblue'), name='real'), 1, 2)
fig.update_layout(height=380, template='plotly_white', showlegend=False,
                  title=f'{T1}: same tag, same historian - the left stretch is a straight line to float32 precision')
fig.show()

print('raw values inside the flagged stretch (constant increment, no curvature):')
print(ts.loc[demo_end - pd.Timedelta(minutes=6):demo_end, T1].to_string())
print('\nraw values in the real stretch (curvature everywhere):')
print(real.tail(7).to_string())

raw values inside the flagged stretch (constant increment, no curvature):
TimeStamp
2023-10-25 16:51:00    37.941429
2023-10-25 16:52:00    37.940315
2023-10-25 16:53:00    37.939201
2023-10-25 16:54:00    37.938084
2023-10-25 16:55:00    37.936970
2023-10-25 16:56:00    37.935852
2023-10-25 16:57:00    37.934738
Freq: min

raw values in the real stretch (curvature everywhere):
TimeStamp
2024-06-01 11:54:00    34.825233
2024-06-01 11:55:00    34.047497
2024-06-01 11:56:00    33.072052
2024-06-01 11:57:00    33.003006
2024-06-01 11:58:00    33.886349
2024-06-01 11:59:00    35.024834
2024-06-01 12:00:00    37.799927
Freq: min


## 3 - Conditions 2 and 3: is the tag actually sitting still?

On the samples that survived Part 2 we slide a window and ask two questions:

- **drift** - fit a straight line through the window; the total change it explains must be
  <= `alpha` x the tag's operating span,
- **variability** - the standard deviation inside the window must be <= `beta` x the operating span.

Because both thresholds are a fraction of the tag's own operating span (p95 - p5), the same two
numbers apply to a level in %, a flow in kg/h and a pressure in barg.

**How strict should alpha and beta be?** That is a real choice, not a detail, so we carry **two**
definitions all the way through instead of asserting one:

| definition | window | alpha | beta | meaning |
|---|---|---|---|---|
| `tight` | 30 min | 2% | 2% | much quieter than this tag normally is |
| `moderate` | 15 min | 5% | 5% | quieter than typical, but not exceptional |

Part 8 shows where each setting sits relative to each tag's *natural* variability and what the choice
does to the vendor comparison. Short version: `tight` sits **below** the median 30-minute movement of
most tags, so it keeps only the quietest few percent of the record.

In [6]:
def rolling_slope(s, w):
    """OLS slope per sample over a trailing window of w samples (vectorised)."""
    x = np.arange(w, dtype=float); xm = x.mean(); sxx = float(((x - xm) ** 2).sum())
    pos = pd.Series(np.arange(len(s), dtype=float), index=s.index)
    sy = s.rolling(w, min_periods=w).sum()
    siy = (pos * s).rolling(w, min_periods=w).sum()
    return (siy - (pos - (w - 1) + xm) * sy) / sxx

def runs_from_mask(mask, min_len):
    """Contiguous True runs of at least min_len samples."""
    m = mask.fillna(False).to_numpy().astype(bool)
    if not m.any():
        return pd.DataFrame(columns=['start', 'end', 'n'])
    d = np.diff(m.astype(np.int8))
    st = np.flatnonzero(d == 1) + 1; en = np.flatnonzero(d == -1)
    if m[0]:
        st = np.r_[0, st]
    if m[-1]:
        en = np.r_[en, len(m) - 1]
    n = en - st + 1; keep = n >= min_len
    return pd.DataFrame(dict(start=mask.index[st[keep]], end=mask.index[en[keep]], n=n[keep]))

CLEAN = {t: ts[t].where(ts[t].notna() & ~QUAL[t]['bad']) for t in TAGS}

def steady_mask(t, W, alpha, beta):
    c = CLEAN[t]
    full = c.notna().rolling(W, min_periods=1).sum() >= W
    drift = rolling_slope(c, W).abs() * W
    sd = c.rolling(W, min_periods=W).std()
    return (full & (drift <= alpha * QUAL[t]['span']) & (sd <= beta * QUAL[t]['span'])).fillna(False)

W = SSD['W']                                           # the tight window, reused by the figures below
DEFS = {'tight': dict(W=30, alpha=0.02, beta=0.02),
        'moderate': dict(W=15, alpha=0.05, beta=0.05)}
S = {k: {t: steady_mask(t, **v) for t in TAGS} for k, v in DEFS.items()}
STEADY = S['tight']                                   # default used by the figures below

rows = []
for t in TAGS:
    r = {'tag': t, 'kept_after_cleaning_pct': round(100 * CLEAN[t].notna().sum()
                                                    / max(int(ts[t].notna().sum()), 1), 1)}
    for k in DEFS:
        rr = runs_from_mask(S[k][t], DEFS[k]['W'])
        r[f'{k}_steady_pct'] = round(100 * S[k][t].mean(), 2)
        r[f'{k}_n_periods'] = len(rr)
        r[f'{k}_median_min'] = int(rr.n.median()) if len(rr) else 0
        r[f'{k}_hours'] = round(rr.n.sum() / 60, 0) if len(rr) else 0
    rows.append(r)
steady_tbl = pd.DataFrame(rows).sort_values('tight_steady_pct', ascending=False)
print('==== steadiness per tag, both definitions ====')
print(steady_tbl.to_string(index=False))

==== steadiness per tag, both definitions ====
           tag  kept_after_cleaning_pct  tight_steady_pct  tight_n_periods  tight_median_min  tight_hours  moderate_steady_pct  moderate_n_periods  moderate_median_min  moderate_hours
 03TIC_1092.OP                    100.0             77.25              892               713      27106.0                78.66                1162                  336         27640.0
 03PIC_1023.OP                     99.5             67.09             5666               121      21866.0                80.54                7841                   50         27385.0
 03FIC_3435.OP                     99.4             64.39             6037                87      21028.0                76.83                6237                   82         26749.0
03HIC_1023A.OP                     99.7             56.99             4599                61      16909.0                72.19                8779                   51         24454.0
  03TI_1108.PV                   

In [7]:
# ---- how much of a NAIVE steady detection is actually synthetic data? ---------------------------
rows = []
for t in TAGS:
    s, q = ts[t], QUAL[t]
    sd = s.rolling(W, min_periods=W).std()
    dr = (s - s.shift(W - 1)).abs()
    naive = ((sd <= SSD['BETA'] * q['span']) & (dr <= SSD['ALPHA'] * q['span']) & s.notna()).fillna(False)
    n = max(int(naive.sum()), 1)
    rows.append(dict(tag=t,
                     naive_steady_pct=round(100 * naive.mean(), 2),
                     of_which_held_pct=round(100 * (naive & q['held']).sum() / n, 1),
                     of_which_rampfill_pct=round(100 * (naive & q['ramp']).sum() / n, 1),
                     of_which_synthetic_pct=round(100 * (naive & q['bad']).sum() / n, 1),
                     cleaned_steady_pct=round(100 * STEADY[t].mean(), 2)))
trap_tbl = pd.DataFrame(rows).sort_values('of_which_synthetic_pct', ascending=False)
print('==== the trap: run the same steadiness test WITHOUT the Part-2 cleaning ====')
print(trap_tbl.to_string(index=False))

==== the trap: run the same steadiness test WITHOUT the Part-2 cleaning ====
           tag  naive_steady_pct  of_which_held_pct  of_which_rampfill_pct  of_which_synthetic_pct  cleaned_steady_pct
 03LIC_1016.PV              0.53                0.2                   99.2                    99.4                0.00
 03LIC_3408.PV              0.72               21.0                   73.0                    94.0                0.03
  02PI_1220.PV              0.59                0.1                   72.5                    72.6                0.22
 03PIC_1013.OP             77.28                0.0                   61.7                    63.6               18.98
 03LIC_1071.PV              1.22                2.6                   43.9                    45.9                0.84
03KM_0152_I.PV              2.95                2.9                   15.9                    18.8                3.47
 03PI_1141A.PV             14.94                3.7                    2.9                

## 4 - Where is the steady-state data? (timelines)

The first figure zooms into one month so the individual windows are visible: green = steady,
orange = historian ramp-fill, red = held value, purple markers = rate-of-change spikes.

The second figure is the whole four-year record as a daily heat strip, one row per tag - it answers
"when do we have steady data and for which tags" at a glance.

In [8]:
def timeline_fig(tag, t0, t1, height=420):
    s = ts.loc[t0:t1, tag]
    q, st = QUAL[tag], STEADY[tag].loc[t0:t1]
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=s.index, y=s.values, mode='lines', name=tag,
                             line=dict(color='#37474f', width=1)))
    bands = [('steady', st, 'rgba(46,160,67,0.30)'),
             ('ramp-fill', q['ramp'].loc[t0:t1], 'rgba(255,152,0,0.35)'),
             ('held', q['held'].loc[t0:t1], 'rgba(211,47,47,0.35)')]
    for name, m, colr in bands:
        for r in runs_from_mask(m, 1).itertuples(index=False):
            fig.add_vrect(x0=r.start, x1=r.end, fillcolor=colr, line_width=0, layer='below')
        fig.add_trace(go.Scatter(x=[None], y=[None], mode='markers', name=name,
                                 marker=dict(size=12, color=colr.replace('0.30', '1').replace('0.35', '1'),
                                             symbol='square')))
    spk = q['spike'].loc[t0:t1]
    if spk.any():
        fig.add_trace(go.Scatter(x=s.index[spk.values], y=s[spk.values], mode='markers',
                                 name='spike', marker=dict(color='purple', size=6, symbol='x')))
    fig.update_layout(height=height, template='plotly_white',
                      title=f'{tag} - {t0} to {t1}   (green = steady state)',
                      yaxis_title=tag, hovermode='x unified')
    return fig

timeline_fig('02FI_1000.PV', '2023-10-20', '2023-11-05').show()
timeline_fig(CONFIG['TARGET'], '2023-10-20', '2023-11-05').show()

In [9]:
# ---- FIGURE: four-year daily steady coverage, one row per tag ------------------------------------
show = [t for t in steady_tbl.tag if STEADY[t].sum() > 0][:24]
daily = pd.DataFrame({t: STEADY[t].resample('1D').mean() * 100 for t in show})
daily = daily.loc[daily.index >= '2022-01-01']
fig = go.Figure(go.Heatmap(z=daily.T.values, x=daily.index, y=daily.columns,
                           colorscale='Greens', zmin=0, zmax=100,
                           colorbar=dict(title='% of day<br>steady')))
fig.update_layout(height=28 * len(show) + 180, template='plotly_white',
                  title='Steady-state coverage per day, per tag (after cleaning)')
fig.show()

print('==== steady-state duration distribution (minutes per uninterrupted period) ====')
rows = []
for t in show:
    r = runs_from_mask(STEADY[t], SSD['MIN_LEN'])
    if len(r):
        rows.append(dict(tag=t, n_periods=len(r), p25=int(r.n.quantile(.25)), median=int(r.n.median()),
                         p75=int(r.n.quantile(.75)), p95=int(r.n.quantile(.95)), max=int(r.n.max())))
print(pd.DataFrame(rows).to_string(index=False))

==== steady-state duration distribution (minutes per uninterrupted period) ====
           tag  n_periods  p25  median  p75  p95   max
 03TIC_1092.OP        892  213     713 1170 7212 53548
 03PIC_1023.OP       5666   53     121  315  773  4789
 03FIC_3435.OP       6037   48      87  199  718 31290
03HIC_1023A.OP       4599   40      61  177  742 42181
  03TI_1108.PV       9390   39      56   93  215  1893
 03TIC_1009.OP       3114   37      51  111  858 23358
 03FIC_3435.PV       8674   37      49   74  166   807
 03TIC_1145.PV       6365   36      45   64  112   469
 03TIC_1009.PV       4120   35      45   67  174  1120
 03TIC_1092.PV       3158   33      39   51   81   353
 03LIC_1071.OP       4441   35      44   61  114   509
  03TI_3112.PV       2409   33      37   46   68   339
 03PIC_1013.OP       2213   44      57   98  598 25087
  03PI_3154.PV       1206   33      39   49   84   248
  03TI_1081.PV        816   32      36   44   60    98
 03TIC_1023.PV       1652   34      41  

## 5 - Condition 4: SP - PV

A controller can sit perfectly still and still not be at steady state - if it is parked away from its
setpoint, something is saturated or in manual. So for controllers we also require
`|PV - SP| <= 5% of span` with **no setpoint move inside the window**.

The historian has no `.SP` column, so setpoints are rebuilt from the `CHANGE` events
(`Description == 'SP'`, forward-filled). That works for base-layer loops the operator sets. It does
**not** work for handles the APC writes - those have no journalled SP at all, which is itself a useful
fact, and the reason this condition is applied only where a real SP trajectory exists.

In [10]:
ev = pd.read_parquet(CONFIG['EVENTS'],
                    columns=['VT_Start', 'Source', 'ConditionName', 'Description',
                             'PrevValue', 'Value', 'EventID'])
ev['VT_Start'] = pd.to_datetime(ev['VT_Start'], errors='coerce')
ev = ev.drop_duplicates(subset=['EventID', 'VT_Start', 'Source', 'Description', 'Value'])
chg = ev[(ev.ConditionName == 'CHANGE') & ev.VT_Start.notna()].copy()
chg['Vnum'] = pd.to_numeric(chg['Value'], errors='coerce')

bases = sorted({t.split('.')[0] for t in TAGS})
cnt = (chg[chg.Source.isin(bases)].groupby(['Source', 'Description']).size()
       .unstack(fill_value=0).reindex(columns=['SP', 'OP', 'MODE'], fill_value=0))
cnt['SP_available'] = cnt['SP'] >= 20
print('==== is a real setpoint trajectory available? (CHANGE events per tag) ====')
print(cnt.sort_values('SP', ascending=False).head(25).to_string())

def recon_sp(base):
    s = chg[(chg.Source == base) & (chg.Description == 'SP')].dropna(subset=['Vnum']).copy()
    if len(s) < 20:
        return None
    s['t'] = s['VT_Start'].dt.floor('1min')
    s = s.sort_values('t').drop_duplicates('t', keep='last').set_index('t')['Vnum']
    return s.reindex(ts.index, method='ffill')

SPREC, rows = {}, []
for base in cnt.index[cnt['SP_available']]:
    col = f'{base}.PV'
    if col not in ts.columns:
        continue
    sp = recon_sp(base)
    if sp is None:
        continue
    SPREC[col] = sp
    pv, span = ts[col], QUAL[col]['span']
    dev = (pv - sp).abs()
    moved = sp.ne(sp.shift()).rolling(W, min_periods=1).max().fillna(0).astype(bool)
    on_sp = ((dev.rolling(W, min_periods=W).max() <= SSD['SP_TOL'] * span) & ~moved).fillna(False)
    st = STEADY[col]
    rows.append(dict(tag=col, sp_events=int(cnt.at[base, 'SP']),
                     sp_cover_pct=round(100 * sp.notna().mean(), 0),
                     med_abs_dev=round(float(dev.median()), 2),
                     on_setpoint_pct=round(100 * on_sp.mean(), 2),
                     steady_pct=round(100 * st.mean(), 2),
                     steady_AND_on_sp_pct=round(100 * (st & on_sp).mean(), 2),
                     steady_but_OFF_sp_pct=round(100 * (st & ~on_sp).mean(), 2)))
    STEADY[col + '__sp'] = (st & on_sp)
print('\n==== SP - PV condition on the loops that have a journalled setpoint ====')
print(pd.DataFrame(rows).to_string(index=False))

==== is a real setpoint trajectory available? (CHANGE events per tag) ====
Description    SP     OP  MODE  SP_available
Source                                      
03TIC_1009   1145     53   412          True
03LIC_3153    805   2384   262          True
03LIC_1071    582   1989   170          True
03LIC_3408    571   1692   223          True
03LIC_1016    530   1394   154          True
03PIC_1023    230      7    68          True
03TIC_1023    138      0    71          True
03TIC_1092      6      1     8         False
03FIC_3435      1  18037   525         False
03HIC_1023A     0    396    62         False
03PIC_1013      0  10461   422         False
03TIC_1145      0     11     0         False
03TI_1005       0      0     0         False
03TI_1081       0      0     0         False
03TI_1108       0      0     0         False
03TI_3112       0      0     0         False



==== SP - PV condition on the loops that have a journalled setpoint ====
          tag  sp_events  sp_cover_pct  med_abs_dev  on_setpoint_pct  steady_pct  steady_AND_on_sp_pct  steady_but_OFF_sp_pct
03LIC_1016.PV        530         100.0         1.12             0.20        0.00                  0.00                   0.00
03LIC_1071.PV        582         100.0         1.05             1.85        0.84                  0.47                   0.37
03LIC_3408.PV        571         100.0         0.45             0.24        0.03                  0.01                   0.02
03TIC_1009.PV       1145         100.0         4.97             2.88       28.98                  1.80                  27.17
03TIC_1023.PV        138         100.0         0.65             9.85       13.75                  4.52                   9.23


In [11]:
# ---- FIGURE: PV vs reconstructed SP with steady windows shaded ----------------------------------
t0, t1 = '2024-03-01', '2024-03-15'
col = CONFIG['TARGET']
fig = go.Figure()
fig.add_trace(go.Scatter(x=ts.loc[t0:t1].index, y=ts.loc[t0:t1, col], mode='lines',
                         name='PV', line=dict(color='#37474f', width=1)))
if col in SPREC:
    fig.add_trace(go.Scatter(x=ts.loc[t0:t1].index, y=SPREC[col].loc[t0:t1], mode='lines',
                             name='SP (rebuilt from events)', line=dict(color='#1e88e5', dash='dash')))
for r in runs_from_mask(STEADY.get(col + '__sp', STEADY[col]).loc[t0:t1], 1).itertuples(index=False):
    fig.add_vrect(x0=r.start, x1=r.end, fillcolor='rgba(46,160,67,0.30)', line_width=0, layer='below')
fig.update_layout(height=420, template='plotly_white', hovermode='x unified',
                  title=f'{col}: PV vs setpoint, green = steady AND on setpoint')
fig.show()

## 6 - Per tag, per pair, or the whole plant?

Steadiness is a per-tag property. For a gain we need it for **two** tags at once - the MV and the CV -
because the vendor number is "CV settled before, CV settled after, MV moved in between". Requiring the
*whole plant* to be steady simultaneously is a different and far stricter thing; the table below shows
how quickly the available data collapses as more tags are required.

In [12]:
core = ['02FI_1000.PV', '03PI_1141A.PV', '03PIC_1013.OP', '03TIC_1092.PV', CONFIG['TARGET']]
core = [c for c in core if c in STEADY]
rows, acc = [], None
for i, t in enumerate(core, 1):
    acc = STEADY[t] if acc is None else (acc & STEADY[t])
    r = runs_from_mask(acc, SSD['MIN_LEN'])
    rows.append(dict(n_tags_required=i, tags=' + '.join(x.split('.')[0] for x in core[:i]),
                     simultaneously_steady_pct=round(100 * acc.mean(), 3),
                     n_periods=len(r), total_hours=round(r.n.sum() / 60, 1) if len(r) else 0.0))
print('==== requiring more tags to be steady at the same time ====')
print(pd.DataFrame(rows).to_string(index=False))

==== requiring more tags to be steady at the same time ====
 n_tags_required                                                          tags  simultaneously_steady_pct  n_periods  total_hours
               1                                                     02FI_1000                     11.802        626        410.2
               2                                        02FI_1000 + 03PI_1141A                      2.419         36         21.5
               3                           02FI_1000 + 03PI_1141A + 03PIC_1013                      0.469         10          6.0
               4              02FI_1000 + 03PI_1141A + 03PIC_1013 + 03TIC_1092                      0.253          3          1.9
               5 02FI_1000 + 03PI_1141A + 03PIC_1013 + 03TIC_1092 + 03LIC_1071                      0.011          0          0.0


## 7 - Does it reproduce the vendor gains?

A vendor gain is measured as: **plant settled -> move the handle -> plant settled again**. Two things
have to be true, and the second one is easy to forget:

1. both ends of the window are at steady state, and
2. **the handle actually moved in between**.

Condition 2 matters more than it looks. If we only impose steadiness, most surviving windows have the
MV sitting at the same value at both ends, so `dMV = 0` while `dCV` still wanders with unmeasured
disturbances. Those rows carry no information about the gain but they do pull the fitted slope towards
zero. Filtering on the size of `dMV` is selection on the regressor, which leaves the least-squares
slope unbiased - it just removes the dead weight.

So four regimes, same estimator, same joint de-confounding, only the rows change:

| regime | rows used |
|---|---|
| `all` | every window in the record (what we reported before) |
| `band` | the customer 5-95%ile operating band |
| `tight` | both ends steady under the strict definition (30 min, 2%) **and** the MV moved >= 5% of its span |
| `moderate` | the same, under the relaxed definition (15 min, 5%) |

`tight` and `moderate` are the regimes built in this notebook, and both are carried through every
pair so the strictness choice is visible rather than assumed. Part 8 shows what that choice is worth.

In [13]:
# ---- band regime (customer 5-95%ile file) --------------------------------------------------------
wl = pd.read_csv(CONFIG['BAND'])
wl['StartTime'] = pd.to_datetime(wl['StartTime'], errors='coerce')
wl['EndTime'] = pd.to_datetime(wl['EndTime'], errors='coerce')
wl = wl.dropna(subset=['StartTime', 'EndTime'])
a = np.sort(wl['StartTime'].values.astype('datetime64[ns]'))
b = wl['EndTime'].values.astype('datetime64[ns]')[np.argsort(wl['StartTime'].values.astype('datetime64[ns]'))]
ii = ts.index.values.astype('datetime64[ns]')
k = np.searchsorted(a, ii, side='right') - 1
seg_band = pd.Series(np.where((k >= 0) & (ii <= b[np.clip(k, 0, len(b) - 1)]), k, np.nan), index=ts.index)

BASE_DECONF = ['02FI_1000.PV', '03PIC_1013.OP', '03TIC_1092.OP']

def inputs_for(mv, cv):
    base = ['03PIC_1013.OP', '03TIC_1092.OP'] if cv == '02FI_1000.PV' else BASE_DECONF
    mi, ci = instr_key(mv), instr_key(cv)
    inp = [mv] + [d for d in base if d in ts.columns and d != mv and d != cv
                  and instr_key(d) != mi and instr_key(d) != ci]
    return list(dict.fromkeys(inp))

def _fit(y, X, cols, mv, nboot=0, block=50, seed=0):
    n = len(y)
    A = np.column_stack([X[c] for c in cols] + [np.ones(n)])
    coef, *_ = np.linalg.lstsq(A, y, rcond=None)
    j = cols.index(mv)
    r2 = 1 - np.var(y - A @ coef) / np.var(y) if np.var(y) > 0 else np.nan
    lo = hi = np.nan
    if nboot and n >= 60:
        rng = np.random.default_rng(seed)
        starts = np.arange(max(1, n - block + 1)); nb = int(np.ceil(n / block)); offs = np.arange(block)
        bs = np.empty(nboot)
        for kk in range(nboot):
            rr = (rng.choice(starts, size=nb)[:, None] + offs[None, :]).ravel(); rr = rr[rr < n][:n]
            cb, *_ = np.linalg.lstsq(A[rr], y[rr], rcond=None); bs[kk] = cb[j]
        lo, hi = np.nanpercentile(bs, [5, 95])
    return float(coef[j]), float(r2), n, float(lo), float(hi)

def gain_diff(cv, mv, H=60, seg=None, endpoints=None, min_move=0.0, thin=20, trim=0.001,
              nboot=0, cap=25000):
    """De-confounded difference gain over an H-minute window.

    endpoints : both ends of the window must satisfy this mask
    min_move  : the MV must change by at least this fraction of its operating span
    """
    inp = inputs_for(mv, cv)
    h = max(1, int(round(H / DT)))
    D = pd.DataFrame({c: ts[c].shift(-h) - ts[c] for c in [cv] + inp})
    if seg is not None:
        D = D[(seg.shift(-h) == seg) & seg.notna()]
    if endpoints is not None:
        D = D[endpoints & endpoints.shift(-h).fillna(False)]
    D = D.replace([np.inf, -np.inf], np.nan).iloc[::thin].dropna()
    if min_move > 0:
        D = D[D[mv].abs() >= min_move * QUAL[mv]['span']]
    if len(D) and trim:
        lo_, hi_ = D[cv].quantile(trim), D[cv].quantile(1 - trim)
        D = D[(D[cv] >= lo_) & (D[cv] <= hi_)]
    if len(D) > cap:                      # keeps the block bootstrap affordable, order preserved
        D = D.iloc[::int(np.ceil(len(D) / cap))]
    cols = [c for c in inp if c == mv or D[c].std() > 0]
    if len(D) < 40 or D[mv].std() == 0:
        return np.nan, np.nan, len(D), np.nan, np.nan
    return _fit(D[cv].to_numpy(), D, cols, mv, nboot=nboot)

def step_events(mv, cv, pair_steady, min_plateau=30, tail=30, max_gap_h=12, min_step_frac=0.03):
    """Count literal step tests in the history: settled plateau -> handle move -> settled plateau."""
    pl = runs_from_mask(STEADY[mv], min_plateau)
    if len(pl) < 5:
        return 0
    pos = pd.Series(np.arange(len(ts)), index=ts.index)
    i1 = pos.reindex(pl['end']).to_numpy(); i0 = np.maximum(i1 - tail + 1, 0)
    amv, acv = ts[mv].to_numpy(), ts[cv].to_numpy()
    cvst = STEADY[cv].to_numpy()
    mmv = np.array([np.nanmean(amv[a:b + 1]) for a, b in zip(i0, i1)])
    ok = np.array([cvst[a:b + 1].mean() >= 0.8 for a, b in zip(i0, i1)])
    gap = (pl['start'].shift(-1) - pl['end']).dt.total_seconds() / 3600
    d = np.abs(np.r_[np.diff(mmv), np.nan])
    good = (gap.to_numpy() > 0) & (gap.to_numpy() <= max_gap_h)         & (d >= min_step_frac * QUAL[mv]['span']) & ok & np.r_[ok[1:], False]
    return int(np.nansum(good))
print('estimators ready')

estimators ready


In [14]:
MIN_MOVE = 0.05                       # the MV must move at least 5% of its operating span
res = []
for p in PAIRS.itertuples(index=False):
    mv, cv = p.our_MV, p.our_CV
    if mv not in ts.columns or cv not in ts.columns or ts[mv].notna().sum() == 0 or ts[cv].notna().sum() == 0:
        continue
    g_all = gain_diff(cv, mv)
    g_band = gain_diff(cv, mv, seg=seg_band)
    row = dict(vendor_MV=p.vendor_MV, vendor_CV=p.vendor_CV, our_MV=mv, our_CV=cv,
               mv_proxy=p.mv_proxy, within_loop=p.within_loop, vendor_gain=p.vendor_gain,
               all_gain=round(g_all[0], 4), all_R2=round(g_all[1], 3),
               band_gain=round(g_band[0], 4),
               ratio_all=round(g_all[0] / p.vendor_gain, 2) if np.isfinite(g_all[0]) else np.nan,
               ratio_band=round(g_band[0] / p.vendor_gain, 2) if np.isfinite(g_band[0]) else np.nan,
               sign_all=(bool(np.sign(g_all[0]) == np.sign(p.vendor_gain))
                         if np.isfinite(g_all[0]) else None))
    for k in DEFS:
        ps = (S[k][mv] & S[k][cv]).fillna(False)
        g = gain_diff(cv, mv, endpoints=ps, min_move=MIN_MOVE, thin=1, nboot=150)
        row[f'{k}_gain'] = round(g[0], 4)
        row[f'{k}_R2'] = round(g[1], 3)
        row[f'{k}_n'] = g[2]
        row[f'{k}_ci90'] = f'[{g[3]:+.3f},{g[4]:+.3f}]' if np.isfinite(g[3]) else 'n/a'
        row[f'ratio_{k}'] = round(g[0] / p.vendor_gain, 2) if np.isfinite(g[0]) else np.nan
        row[f'sign_{k}'] = bool(np.sign(g[0]) == np.sign(p.vendor_gain)) if np.isfinite(g[0]) else None
        row[f'{k}_pair_steady_pct'] = round(100 * ps.mean(), 2)
    row['n_step_tests'] = step_events(mv, cv, None)
    res.append(row)
vres = pd.DataFrame(res).sort_values(['within_loop', 'our_CV', 'our_MV']).reset_index(drop=True)

show = ['our_MV', 'our_CV', 'mv_proxy', 'vendor_gain', 'all_gain', 'band_gain',
        'tight_gain', 'tight_R2', 'tight_n', 'moderate_gain', 'moderate_R2', 'moderate_n',
        'ratio_all', 'ratio_band', 'ratio_tight', 'ratio_moderate']
print('==== PAIRWISE VENDOR VALIDATION - every resolvable MV -> CV cell ====')
print('ratio = derived / vendor;  1.00 = exact magnitude match\n')
print(vres[show].to_string(index=False))

==== PAIRWISE VENDOR VALIDATION - every resolvable MV -> CV cell ====
ratio = derived / vendor;  1.00 = exact magnitude match

       our_MV         our_CV     mv_proxy  vendor_gain  all_gain  band_gain  tight_gain  tight_R2  tight_n  moderate_gain  moderate_R2  moderate_n  ratio_all  ratio_band  ratio_tight  ratio_moderate
03FIC_3435.OP   02FI_1000.PV                    0.0066    0.0297     0.0222      0.0252     0.542      924         0.0297        0.558       20654       4.50        3.37         3.82            4.51
03FIC_3435.OP   02PI_1220.PV                   -0.0766   -0.0496    -0.0290         NaN       NaN        0        -0.0388        0.386        3511       0.65        0.38          NaN            0.51
03FIC_3435.PV   02PI_1220.PV        FF=PV      -0.0400   -0.0000    -0.0000         NaN       NaN        5        -0.0000        0.342        6109       0.00        0.00          NaN            0.00
 02FI_1000.PV 03HIC_1023A.OP        FF=PV       9.4064    9.1195     6.6761  

In [15]:
ok = vres[~vres.within_loop].copy()

def _sc(col):
    s = ok[col].dropna().astype(bool)
    return f'{int(s.sum())}/{len(s)}'
print('sign-match vs vendor:  all data', _sc('sign_all'),
      '|  tight', _sc('sign_tight'), '|  moderate', _sc('sign_moderate'))

# a gain fitted with no explanatory power is not evidence either way
ident = ok[(ok.moderate_R2 >= 0.10) | (ok.tight_R2 >= 0.10)]
print(f'\nmedian |derived/vendor - 1|        all pairs (n={len(ok)})   identifiable (n={len(ident)})')
for lab, rc in [('all data', 'ratio_all'), ('5-95%ile band', 'ratio_band'),
                ('steady tight', 'ratio_tight'), ('steady moderate', 'ratio_moderate')]:
    a = (ok[rc] - 1).abs().dropna().median()
    b = (ident[rc] - 1).abs().dropna().median()
    print(f'  {lab:20s}            {a:6.2f}                {b:6.2f}')

print('\n==== per-pair scorecard (identifiable pairs only) ====')
sc = ident[['our_MV', 'our_CV', 'vendor_gain', 'all_gain', 'band_gain', 'tight_gain',
            'moderate_gain', 'tight_R2', 'moderate_R2', 'tight_n', 'moderate_n',
            'ratio_all', 'ratio_band', 'ratio_tight', 'ratio_moderate']].copy()
sc['best_regime'] = sc[['ratio_all', 'ratio_band', 'ratio_tight', 'ratio_moderate']].apply(
    lambda r: (r - 1).abs().idxmin().replace('ratio_', '') if r.notna().any() else '-', axis=1)
sc['best_ratio'] = sc[['ratio_all', 'ratio_band', 'ratio_tight', 'ratio_moderate']].apply(
    lambda r: r[(r - 1).abs().idxmin()] if r.notna().any() else np.nan, axis=1)
print(sc.to_string(index=False))
print('\nwhich regime lands closest to vendor, per pair:')
print(sc['best_regime'].value_counts().to_string())

print('\nliteral step tests in the history (settled -> handle move -> settled), per MV:')
print(ok[ok.n_step_tests > 0][['our_MV', 'our_CV', 'n_step_tests']]
      .sort_values('n_step_tests', ascending=False).head(10).to_string(index=False))

with pd.ExcelWriter(f'{OUT}/steady_state_gain_validation.xlsx') as xl:
    vres.to_excel(xl, sheet_name='pairwise_vendor_validation', index=False)
    sc.to_excel(xl, sheet_name='scorecard_identifiable', index=False)
    qual_tbl.to_excel(xl, sheet_name='data_quality', index=False)
    steady_tbl.to_excel(xl, sheet_name='steadiness_per_tag', index=False)
    trap_tbl.to_excel(xl, sheet_name='naive_vs_cleaned', index=False)

per = []
for k in DEFS:
    for t in TAGS:
        r = runs_from_mask(S[k][t], DEFS[k]['W'])
        if len(r):
            r.insert(0, 'tag', t); r.insert(0, 'definition', k)
            per.append(r)
periods = pd.concat(per, ignore_index=True)
periods.to_csv(f'{OUT}/steady_state_periods.csv', index=False)
print(f'\nsaved -> {OUT}/steady_state_periods.csv  ({len(periods):,} steady periods, both definitions)')
print(f'saved -> {OUT}/steady_state_gain_validation.xlsx')

sign-match vs vendor:  all data 20/24 |  tight 15/21 |  moderate 20/24

median |derived/vendor - 1|        all pairs (n=24)   identifiable (n=15)
  all data                          0.72                  0.39
  5-95%ile band                     0.57                  0.54
  steady tight                      0.93                  0.96
  steady moderate                   0.66                  0.60

==== per-pair scorecard (identifiable pairs only) ====
       our_MV         our_CV  vendor_gain  all_gain  band_gain  tight_gain  moderate_gain  tight_R2  moderate_R2  tight_n  moderate_n  ratio_all  ratio_band  ratio_tight  ratio_moderate best_regime  best_ratio
03FIC_3435.OP   02FI_1000.PV       0.0066    0.0297     0.0222      0.0252         0.0297     0.542        0.558      924       20654       4.50        3.37         3.82            4.51        band        3.37
03FIC_3435.OP   02PI_1220.PV      -0.0766   -0.0496    -0.0290         NaN        -0.0388       NaN        0.386        0     


saved -> /home/h604827/ControlActions/RESULTS/steady_state_detection/steady_state_periods.csv  (452,649 steady periods, both definitions)
saved -> /home/h604827/ControlActions/RESULTS/steady_state_detection/steady_state_gain_validation.xlsx


In [16]:
# ---- FIGURE: how the derived gain moves as we demand more MV movement --------------------------
SWEEP = [0.0, 0.01, 0.02, 0.05, 0.10]
pick = ident.sort_values('tight_R2', ascending=False).drop_duplicates(subset=['our_CV']).head(6)
fig = make_subplots(rows=2, cols=3,
                    subplot_titles=[f'{r.our_MV} -> {r.our_CV}' for r in pick.itertuples()])
for i, r in enumerate(pick.itertuples(index=False)):
    rr, cc = i // 3 + 1, i % 3 + 1
    for k, colr in [('tight', '#1e88e5'), ('moderate', '#43a047')]:
        ps = (S[k][r.our_MV] & S[k][r.our_CV]).fillna(False)
        gs, ns = [], []
        for m in SWEEP:
            g, _r2, n, _, _ = gain_diff(r.our_CV, r.our_MV, endpoints=ps, min_move=m, thin=1)
            gs.append(g); ns.append(n)
        fig.add_trace(go.Scatter(x=[100 * s for s in SWEEP], y=gs, mode='lines+markers',
                                 line=dict(color=colr), name=k, legendgroup=k,
                                 text=[f'n={n}' for n in ns], showlegend=(i == 0)), rr, cc)
    fig.add_hline(y=r.vendor_gain, line=dict(color='crimson', dash='dash'), row=rr, col=cc)
    fig.update_xaxes(title_text='min MV move (% of span)', row=rr, col=cc)
fig.update_layout(height=650, template='plotly_white',
                  title='Derived gain vs required MV excitation (red dashed = vendor value)')
fig.show()

## 8 - Are we being too strict?

The thresholds are only meaningful next to each tag's **natural** movement. The first table shows the
distribution of the 30-minute drift and standard deviation that each tag exhibits anyway, as a
percentage of its own span. If our threshold sits below the median of that distribution, we are
keeping only unusually quiet windows - which is a defensible definition of steady state, but it has a
price in sample size, and the price has to be visible.

The second table sweeps window length and thresholds and reports what it does to the vendor
comparison, so the choice is made against evidence rather than taste.

In [ ]:
# ---- where do our thresholds sit relative to each tag's own behaviour? -------------------------
rows = []
for t in TAGS:
    c = CLEAN[t]
    sd = c.rolling(30, min_periods=30).std() / QUAL[t]['span'] * 100
    dr = rolling_slope(c, 30).abs() * 30 / QUAL[t]['span'] * 100
    rows.append(dict(tag=t, span=round(QUAL[t]['span'], 3),
                     sd_p25=round(float(sd.quantile(.25)), 2), sd_p50=round(float(sd.median()), 2),
                     sd_p75=round(float(sd.quantile(.75)), 2),
                     drift_p25=round(float(dr.quantile(.25)), 2), drift_p50=round(float(dr.median()), 2),
                     drift_p75=round(float(dr.quantile(.75)), 2)))
nat = pd.DataFrame(rows).sort_values('sd_p50', ascending=False)
print('==== natural 30-minute movement per tag, as % of the tag span ====')
print('tight uses 2%, moderate uses 5% - compare against the p25/p50 columns\n')
print(nat.to_string(index=False))
print('\ntags whose MEDIAN 30-min std already exceeds the tight 2% threshold:',
      int((, 'of', le

==== natural 30-minute movement per tag, as % of the tag span ====
tight uses 2%, moderate uses 5% - compare against the p25/p50 columns

           tag      span  sd_p25  sd_p50  sd_p75  drift_p25  drift_p50  drift_p75
 03LIC_1016.PV    11.107    9.21   13.45   18.69       2.40       5.35      10.30
  02PI_1220.PV     1.056    5.82    8.50   12.81       7.01      15.72      30.14
 03LIC_1071.PV    12.956    4.49    8.43   15.03       3.34       7.59      15.09
03KM_0152_I.PV    16.994    3.49    5.47    7.96       2.79       6.83      14.58
 03LIC_3408.PV     8.170    4.18    5.13    6.98       2.10       4.73       9.70
 03LIC_1016.OP    36.740    2.35    3.37    4.60       1.17       2.64       5.20
 03TIC_1023.PV     4.176    2.09    3.21    5.32       1.86       5.00      12.04
 03LIC_3153.OP    32.496    2.59    3.19    4.77       1.63       4.00      10.11
  03PI_3154.PV     1.116    1.84    2.70    4.08       1.99       4.56       9.05
  02FI_1000.PV     2.218    2.06    2.70  

In [18]:
# ---- sweep the definition and watch the vendor comparison ---------------------------------------
GRID = [(15, 0.02, 0.02), (15, 0.05, 0.05), (15, 0.10, 0.10),
        (30, 0.02, 0.02), (30, 0.05, 0.05), (30, 0.10, 0.10),
        (60, 0.05, 0.05), (60, 0.10, 0.10)]
anchors = [(r.our_MV, r.our_CV, r.vendor_gain) for r in ident.itertuples(index=False)]
rows = []
for w, a, b in GRID:
    Sw = {}
    for mv, cv, _ in anchors:
        for t in (mv, cv):
            if t not in Sw:
                Sw[t] = steady_mask(t, w, a, b)
    for mm in (0.0, MIN_MOVE):
        err, r2s, ns = [], [], []
        for mv, cv, vg in anchors:
            g = gain_diff(cv, mv, endpoints=(Sw[mv] & Sw[cv]).fillna(False), min_move=mm, thin=1)
            if np.isfinite(g[0]):
                err.append(abs(g[0] / vg - 1)); r2s.append(g[1]); ns.append(g[2])
        rows.append(dict(window_min=w, alpha=a, beta=b, min_MV_move=mm, pairs=len(err),
                         steady_pct_feed=round(100 * Sw['02FI_1000.PV'].mean(), 1),
                         median_abs_ratio_err=round(float(np.median(err)), 2),
                         median_R2=round(float(np.median(r2s)), 3),
                         median_n=int(np.median(ns))))
sens = pd.DataFrame(rows)
base_err = (ident['ratio_all'] - 1).abs().median()
print('==== strictness sweep on the identifiable anchors ====')
print(f'reference: no steady filter at all gives median |ratio-1| = {base_err:.2f}\n')
print(sens.to_string(index=False))

==== strictness sweep on the identifiable anchors ====
reference: no steady filter at all gives median |ratio-1| = 0.39

 window_min  alpha  beta  min_MV_move  pairs  steady_pct_feed  median_abs_ratio_err  median_R2  median_n
         15   0.02  0.02         0.00     15             19.4                  0.67      0.197     19386
         15   0.02  0.02         0.05     15             19.4                  0.71      0.438      3255
         15   0.05  0.05         0.00     15             55.3                  0.63      0.237     24207
         15   0.05  0.05         0.05     15             55.3                  0.60      0.386     20749
         15   0.10  0.10         0.00     15             79.0                  0.48      0.350     24591
         15   0.10  0.10         0.05     15             79.0                  0.50      0.369     23632
         30   0.02  0.02         0.00     15             11.8                  0.91      0.195      6854
         30   0.02  0.02         0.05  

## 9 - Reading the result

- `ratio_*` close to **1.00** means the derived gain reproduces the vendor number in magnitude, not
  just in sign. `best_regime` in the scorecard says which regime got closest for that pair.
- `*_R2` is the guard: a gain fitted with no explanatory power is not a gain, and forcing one out of a
  parked valve would be curve-fitting. Only the identifiable block is evidence.
- `*_n` and `*_pair_steady_pct` say how much data the regime actually had. A pair with a tiny `n` is
  untested, not a miss.
- `n_step_tests` counts literal settled -> move -> settled events in four years. For the strongest
  anchors it is near zero, which is why the vendor step test cannot simply be replayed.
- Integrating tags (levels) cannot have a steady-state gain by construction; Part 3 shows they are
  almost never steady, which is the data-side confirmation of that, not a failure of the detector.

**Where this leaves us.** The steady-state definition is now a stated, reproducible rule, and Part 9
shows exactly how strict it is relative to each tag's own behaviour rather than asserting a setting.
Strictness is a genuine trade-off: `tight` isolates the cleanest data and reproduces the best anchor
(feed -> `03PI_1141A`) at close to vendor magnitude, but it starves several pairs; `moderate` keeps far
more data and does better on aggregate. Neither wins everywhere, because the remaining mismatch on
pairs like `03PIC_1013.OP -> 03PI_1141A` is driven by operating point and closed-loop rejection, not by
transient contamination. Combining this filter with feed-rate stratification is the next cut.